In [0]:
# 02_bronze/2_bronze_transform.py
#  BRONZE: Creación de IDs numéricos

print("="*50)
print(" BRONZE: Creación de IDs numéricos")
print("="*50)

from pyspark.sql.functions import col, to_date, when, lit
from pyspark.sql.types import IntegerType

df_raw = spark.table("capa_raw.coffee_sales_raw")

# PASO 1: Estandarizar tipos
df = df_raw \
    .withColumn("hour_of_day", col("hour_of_day").cast("int")) \
    .withColumn("cash_type", col("cash_type").cast("string")) \
    .withColumn("money", col("money").cast("double")) \
    .withColumn("coffee_name", col("coffee_name").cast("string")) \
    .withColumn("Time_of_Day", col("Time_of_Day").cast("string")) \
    .withColumn("Weekday", col("Weekday").cast("string")) \
    .withColumn("Month_name", col("Month_name").cast("string")) \
    .withColumn("Weekdaysort", col("Weekdaysort").cast("int")) \
    .withColumn("Monthsort", col("Monthsort").cast("int")) \
    .withColumn("Date", to_date(col("Date"), "yyyy-MM-dd"))

# =====================================================
# PASO 2: CREAR IDs CON DICCIONARIOS (SIN AMBIGÜEDAD)
# =====================================================

# 2.1 ID para Producto (1, 2, 3, 4, 5...)
productos = sorted([row.coffee_name for row in df.select("coffee_name").distinct().collect()])
productos_dict = {nombre: idx + 1 for idx, nombre in enumerate(productos)}

from pyspark.sql.functions import udf
@udf(returnType=IntegerType())
def get_product_id(nombre):
    return productos_dict.get(nombre, 0)

df = df.withColumn("Product_ID", get_product_id(col("coffee_name")))

# 2.2 ID para Fecha (1, 2, 3, 4, 5...)
fechas = sorted([row.Date for row in df.select("Date").distinct().collect()])
fechas_dict = {fecha: idx + 1 for idx, fecha in enumerate(fechas)}

@udf(returnType=IntegerType())
def get_date_id(fecha):
    return fechas_dict.get(fecha, 0)

df = df.withColumn("Date_ID", get_date_id(col("Date")))

# 2.3 ID para Hora (SOLO 3 VALORES: 1=Morning, 2=Afternoon, 3=Night)

df = df.withColumn("Time_Period",
    when(col("hour_of_day") < 12, "Morning")
    .when(col("hour_of_day") < 18, "Afternoon")
    .otherwise("Night"))

time_periods = ["Morning", "Afternoon", "Night"]
time_dict = {periodo: idx + 1 for idx, periodo in enumerate(time_periods)}

@udf(returnType=IntegerType())
def get_time_id(periodo):
    return time_dict.get(periodo, 0)

df = df.withColumn("Time_ID", get_time_id(col("Time_Period")))

# 2.4 ID para Día de semana (1=Monday, 2=Tuesday...)
dias = sorted([row.Weekday for row in df.select("Weekday", "Weekdaysort").distinct().orderBy("Weekdaysort").collect()])
dias_dict = {dia: idx + 1 for idx, dia in enumerate(dias)}

@udf(returnType=IntegerType())
def get_day_id(dia):
    return dias_dict.get(dia, 0)

df = df.withColumn("Day_ID", get_day_id(col("Weekday")))

# 2.5 ID para Mes (1=January, 2=February...)
meses = [row.Month_name for row in df.select("Month_name", "Monthsort").distinct().orderBy("Monthsort").collect()]
meses_dict = {mes: idx + 1 for idx, mes in enumerate(meses)}

@udf(returnType=IntegerType())
def get_month_id(mes):
    return meses_dict.get(mes, 0)

df = df.withColumn("Month_ID", get_month_id(col("Month_name")))

# 2.6 ID para Temporada (1=Spring, 2=Summer, 3=Fall, 4=Winter)
df = df.withColumn("Season",
    when(col("Monthsort").isin([3, 4, 5]), "Spring")
    .when(col("Monthsort").isin([6, 7, 8]), "Summer")
    .when(col("Monthsort").isin([9, 10, 11]), "Fall")
    .otherwise("Winter"))

temporadas = ["Spring", "Summer", "Fall", "Winter"]
temporadas_dict = {temp: idx + 1 for idx, temp in enumerate(temporadas)}

@udf(returnType=IntegerType())
def get_season_id(temp):
    return temporadas_dict.get(temp, 0)

df = df.withColumn("Season_ID", get_season_id(col("Season")))

# 2.7 ID para Método de Pago (1=card, 2=cash)
pagos = sorted([row.cash_type for row in df.select("cash_type").distinct().collect()])
pagos_dict = {pago: idx + 1 for idx, pago in enumerate(pagos)}

@udf(returnType=IntegerType())
def get_payment_id(pago):
    return pagos_dict.get(pago, 0)

df = df.withColumn("Payment_ID", get_payment_id(col("cash_type")))

# PASO 3: Guardar en Bronze
df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("capa_bronze.coffee_sales_bronze")

print(f" {df.count()} registros en Bronze")

print("\n MAPEO DE IDs:")
print(f"  Productos: {productos_dict}")
print(f"  Fechas: {len(fechas_dict)} fechas únicas")
print(f"  Horas (SOLO 3): {time_dict}")
print(f"  Días: {dias_dict}")
print(f"  Meses: {meses_dict}")
print(f"  Temporadas: {temporadas_dict}")
print(f"  Pagos: {pagos_dict}")

print("\n Ejemplo de datos con IDs:")
display(df.select("coffee_name", "Product_ID", "Date", "Date_ID", "Time_Period", "Time_ID", "Weekday", "Day_ID", "cash_type", "Payment_ID").limit(10))

print("="*50)
print(" BRONZE COMPLETADO")
print("="*50)